# Análisis de resultados

Este notebook **no ejecuta experimentos** ni realiza llamadas al LLM.
Solo **lee** artefactos ya generados por el runner y prepara la base para el análisis (tablas/figuras).

## Contrato de datos (alineación con el runner)

### Entradas esperadas
- `results/experiment_results.csv`
- `results/run_metadata.json`
- `results/trajectories/*.json`

### Columnas mínimas requeridas en el CSV
- Identidad: `instance`, `instance_size`, `variant`, `algorithm`, `seed`, `run`
- Estado: `success`, `failed`
- Métricas: `total_cost`, `num_courses`, `elapsed_time`, `llm_calls`
- Evaluación LLM: `llm_evaluation_score`, `llm_evaluation_nota`
- Trayectoria: `trajectory_found`, `trajectory_path`

### Reglas de inclusión (definiciones únicas)
- `is_valid_run = (failed == 0)`
- `is_success = (success == 1) & is_valid_run`
- **Métricas de performance**: agregados en subset `is_success == True`
- **Métricas de robustez**: agregados en subset `is_valid_run == True`

In [33]:
# Imports (solo análisis; sin llamadas externas)
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

In [34]:
# 0.1 Rutas: verificar que los archivos/directorios de entrada existen
from pathlib import Path

# El notebook vive en report/, así que el root del repo es el padre de esta carpeta
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "report" else Path.cwd().resolve()

RESULTS_DIR = REPO_ROOT / "results"
CSV_PATH = RESULTS_DIR / "experiment_results.csv"
METADATA_PATH = RESULTS_DIR / "run_metadata.json"
TRAJECTORIES_DIR = RESULTS_DIR / "trajectories"

missing = []
for p in [RESULTS_DIR, CSV_PATH, METADATA_PATH, TRAJECTORIES_DIR]:
    if not p.exists():
        missing.append(str(p))

if missing:
    raise FileNotFoundError("Faltan entradas requeridas:\n- " + "\n- ".join(missing))

trajectory_files = sorted(TRAJECTORIES_DIR.glob("*.json"))
if not trajectory_files:
    raise FileNotFoundError(
        f"No se encontraron trajectories JSON en: {TRAJECTORIES_DIR}\n"
        "Se esperaba al menos un archivo *.json."
    )

print("Entradas OK")
print(f"- Repo root: {REPO_ROOT}")
print(f"- CSV: {CSV_PATH}")
print(f"- Metadata: {METADATA_PATH}")
print(f"- Trajectories: {TRAJECTORIES_DIR} ({len(trajectory_files)} archivos)")

Entradas OK
- Repo root: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm
- CSV: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\results\experiment_results.csv
- Metadata: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\results\run_metadata.json
- Trajectories: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\results\trajectories (1500 archivos)


In [35]:
# Cargar y mostrar metadata de ejecución
with METADATA_PATH.open("r", encoding="utf-8") as f:
    run_metadata = json.load(f)

print("run_metadata.json (resumen)")
for k in sorted(run_metadata.keys()):
    v = run_metadata[k]
    # evitar imprimir blobs enormes si aparecieran
    text = str(v)
    if len(text) > 300:
        text = text[:300] + "…"
    print(f"- {k}: {text}")

run_metadata.json (resumen)
- controlled_environment_statement: Ejecutado en la misma máquina y sin cargas pesadas en paralelo (declaración del ejecutor).
- cpu_info: Intel64 Family 6 Model 142 Stepping 12, GenuineIntel
- endpoint_local: http://localhost:11434
- git_commit: 8479bae1c9022cec93c0335c01fb3effc6feadff
- machine_arch: AMD64
- max_retries: 1
- ollama_model: qwen2.5:3b
- os_name: Windows
- os_version: 10.0.26200
- python_version: 3.14.0
- request_timeout: 120
- temperature: 0.1
- timestamp_iso: 2026-05-27T15:49:41-0400


In [36]:
# 0.2 Columnas mínimas: validar que el CSV tenga el schema requerido
# Nota importante: esta celda SIEMPRE vuelve a leer el CSV desde disco.
import os
print(f"Leyendo CSV desde: {CSV_PATH}")
print(f"- Existe: {CSV_PATH.exists()}")
if CSV_PATH.exists():
    st = CSV_PATH.stat()
    print(f"- Tamaño (bytes): {st.st_size}")
    print(f"- mtime: {st.st_mtime}")

df_raw = pd.read_csv(CSV_PATH)

REQUIRED_COLUMNS = [
    # Identidad
    "instance", "instance_size", "variant", "algorithm", "seed", "run",
    # Estado
    "success", "failed",
    # Métricas
    "total_cost", "num_courses", "elapsed_time", "llm_calls",
    # Evaluación LLM
    "llm_evaluation_score", "llm_evaluation_nota",
    # Trayectoria
    "trajectory_found", "trajectory_path",
]

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
if missing_cols:
    raise ValueError(
        "El CSV no contiene todas las columnas mínimas requeridas.\n"
        + "Faltan:\n- " + "\n- ".join(missing_cols) + "\n"
        + "Columnas presentes (para debug):\n- " + "\n- ".join(map(str, df_raw.columns.tolist()))
    )

# --- Chequeo de cobertura del experimento (para evitar análisis inválidos con datos de validación) ---
n_rows = len(df_raw)
n_instances = df_raw["instance"].nunique(dropna=False)
n_variants = df_raw["variant"].nunique(dropna=False)
n_algorithms = df_raw["algorithm"].nunique(dropna=False)
n_seeds = df_raw["seed"].nunique(dropna=False)

print(f"CSV OK: {n_rows} filas, {len(df_raw.columns)} columnas")
print("Cobertura detectada:")
print(f"- instancias únicas:  {n_instances}")
print(f"- variantes únicas:   {n_variants}")
print(f"- algoritmos únicos:  {n_algorithms}")
print(f"- semillas únicas:    {n_seeds}")

# Heurística: si parece un dataset de validación (p.ej., 4 instancias y 1 semilla), se aborta.
# Umbrales conservadores: el experimento completo normalmente tiene muchísimas más filas y semillas.
is_smoke_like = (n_instances <= 5) and (n_seeds <= 1) and (n_rows <= 100)
if is_smoke_like:
    raise RuntimeError(
        "Dataset demasiado pequeño (parece una ejecución de validación/smoke).\n\n"
        "Esto invalida tablas, gráficos y pruebas estadísticas (n insuficiente).\n\n"
        "Qué hacer:\n"
        "1) Asegúrate de que results/experiment_results.csv sea el del experimento completo.\n"
        "2) Reinicia el kernel y ejecuta este notebook desde la primera celda.\n\n"
        "Pista: el experimento completo debería tener muchas más instancias y varias semillas."
    )

Leyendo CSV desde: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\results\experiment_results.csv
- Existe: True
- Tamaño (bytes): 255336
- mtime: 1779917784.2906606
CSV OK: 1500 filas, 18 columnas
Cobertura detectada:
- instancias únicas:  30
- variantes únicas:   4
- algoritmos únicos:  3
- semillas únicas:    5


In [37]:
# 0.3 Reglas de inclusión: crear columnas derivadas con nombres exactos
df = df_raw.copy()

# Normalización ligera para evitar problemas comunes
# - Convertir vacíos a NaN
df = df.replace({"": np.nan, "None": np.nan})

# Asegurar tipos (sin ser agresivo; lo importante aquí es el contrato)
for col in ["success", "failed"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

df["is_valid_run"] = df["failed"] == 0
df["is_success"] = (df["success"] == 1) & df["is_valid_run"]

# Resumen rápido del contrato
print("Resumen:")
print("- is_valid_run = (failed == 0)")
print("- is_success = (success == 1) & is_valid_run")
print()
print(f"Filas is_valid_run=True: {int(df['is_valid_run'].sum())} / {len(df)}")
print(f"Filas is_success=True:   {int(df['is_success'].sum())} / {len(df)}")

Resumen:
- is_valid_run = (failed == 0)
- is_success = (success == 1) & is_valid_run

Filas is_valid_run=True: 1500 / 1500
Filas is_success=True:   1305 / 1500


### Definiciones que se usarán en el resto del notebook
- **Métricas de performance**: se calculan usando el subset `df[df['is_success'] == True]`
- **Métricas de robustez**: se calculan usando el subset `df[df['is_valid_run'] == True]`

## Carga, validación y limpieza (reproducible)

> Esta sección prepara un dataset **limpio** a partir del CSV original y lo exporta a `report/tables/clean_results.csv`.

> Reglas:
- Si falta un archivo de entrada o una columna mínima, el notebook debe fallar **temprano** con un mensaje claro.

In [38]:
# 2.1 Load + 2.2 Validate (con error temprano y mensajes claros)

# Rutas de salida del análisis
REPORT_DIR = REPO_ROOT / "report"
TABLES_DIR = REPORT_DIR / "tables"
CLEAN_CSV_PATH = TABLES_DIR / "clean_results.csv"

# Asegurar carpetas de salida
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Cargar (de nuevo, de forma explícita y reproducible)
df_raw = pd.read_csv(CSV_PATH)

# Validar columnas mínimas
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
if missing_cols:
    raise ValueError(
        "El CSV no contiene todas las columnas mínimas requeridas.\n"
        + "Faltan:\n- " + "\n- ".join(missing_cols) + "\n\n"
        + "Sugerencia: revisa el runner/export y vuelve a generar el CSV.\n"
        + "Columnas presentes:\n- " + "\n- ".join(map(str, df_raw.columns.tolist()))
    )

print("Carga y validación: OK")
print(f"- Filas: {len(df_raw)}")
print(f"- Columnas: {len(df_raw.columns)}")
print(f"- Output esperado: {CLEAN_CSV_PATH}")

Carga y validación: OK
- Filas: 1500
- Columnas: 18
- Output esperado: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\clean_results.csv


In [39]:
# 2.3 Clean: dtypes + NaN + columnas derivadas EXACTAS

df = df_raw.copy()

# Normalizar vacíos a NaN
df = df.replace({"": np.nan, "None": np.nan})

# Dtypes: success/failed -> int
for col in ["success", "failed"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# Dtypes: métricas numéricas -> numéricas (NaN si no parsea)
for col in ["total_cost", "num_courses", "elapsed_time", "llm_calls"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Columnas derivadas EXACTAS
df["is_valid_run"] = df["failed"] == 0
df["is_success"] = (df["success"] == 1) & df["is_valid_run"]
df["has_llm_eval"] = df["llm_evaluation_nota"].notna()
df["is_llm_variant"] = df["variant"].isin(["B", "C", "D"])

print("Limpieza: OK")
print("Resumen de filas:")
print(f"- is_valid_run=True: {int(df['is_valid_run'].sum())} / {len(df)}")
print(f"- is_success=True:   {int(df['is_success'].sum())} / {len(df)}")
print(f"- has_llm_eval=True: {int(df['has_llm_eval'].sum())} / {len(df)}")
print(f"- is_llm_variant=True: {int(df['is_llm_variant'].sum())} / {len(df)}")

Limpieza: OK
Resumen de filas:
- is_valid_run=True: 1500 / 1500
- is_success=True:   1305 / 1500
- has_llm_eval=True: 310 / 1500
- is_llm_variant=True: 1125 / 1500

Resumen de filas:
- is_valid_run=True: 1500 / 1500
- is_success=True:   1305 / 1500
- has_llm_eval=True: 310 / 1500
- is_llm_variant=True: 1125 / 1500


In [40]:
# 2.4 Export: guardar dataset limpio (reproducible)

df.to_csv(CLEAN_CSV_PATH, index=False, encoding="utf-8")

print("Export: OK")
print(f"- Archivo generado: {CLEAN_CSV_PATH}")
print(f"- Tamaño (bytes): {CLEAN_CSV_PATH.stat().st_size}")

# Vista rápida
df.head(3)

Export: OK
- Archivo generado: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\clean_results.csv
- Tamaño (bytes): 283753


,instance,instance_size,variant,algorithm,seed,run,success,total_cost,num_courses,elapsed_time,...,llm_evaluation_nota,trajectory_path,trajectory_found,variant_type,error_message,failed,is_valid_run,is_success,has_llm_eval,is_llm_variant
0,synthetic_10_courses_01.json,small,A,greedy,1,1,0,18,6,0.000405,...,NaN,trajectories\synthetic_10_courses_01.json__A__...,0,oracle,NaN,0,True,False,False,False
1,synthetic_10_courses_01.json,small,A,greedy,2,2,0,18,6,0.000077,...,NaN,trajectories\synthetic_10_courses_01.json__A__...,0,oracle,NaN,0,True,False,False,False
2,synthetic_10_courses_01.json,small,A,greedy,3,3,0,18,6,0.000067,...,NaN,trajectories\synthetic_10_courses_01.json__A__...,0,oracle,NaN,0,True,False,False,False


## Estadística descriptiva + tablas exportadas

> En esta sección calculamos estadísticas en dos vistas:
- **performance**: subset `is_success == True`
- **robustez**: subset `is_valid_run == True`

> Todas las tablas se exportan a `report/tables/` e incluyen una columna `subset` para dejarlo explícito.

In [62]:
# Helpers de agregación (reproducibles y fáciles de auditar)

METRIC_COLS = ["total_cost", "num_courses", "elapsed_time"]
GROUP_COLS_VARIANT = ["variant"]
GROUP_COLS_SIZE = ["instance_size"]
GROUP_COLS_VARIANT_SIZE = ["variant", "instance_size"]

def _agg_table(df_in: pd.DataFrame, group_cols: list[str], subset_name: str) -> pd.DataFrame:
    """Devuelve una tabla con media/mediana/std para métricas + tasa de éxito.

    - subset_name: 'performance' o 'robustez' (se agrega como columna).
    """
    if subset_name not in {"performance", "robustez"}:
        raise ValueError(f"subset_name inválido: {subset_name}")

    grouped = df_in.groupby(group_cols, dropna=False)

    # Métricas (mean/median/std)
    out = grouped[METRIC_COLS].agg(["mean", "median", "std"]).reset_index()

    # Aplanar nombres de columnas multiíndice: total_cost_mean, etc.
    out.columns = [
        "_".join([c for c in col if c]) if isinstance(col, tuple) else str(col)
        for col in out.columns
    ]

    # Tasa de éxito (siempre sobre df_in, pero depende del subset)
    # - en performance debería dar 1.0 casi siempre (porque filtramos is_success), pero se incluye igual para consistencia
    success_rate = grouped["is_success"].mean().reset_index().rename(columns={"is_success": "success_rate"})
    out = out.merge(success_rate, on=group_cols, how="left")

    # n_rows: evitar desalineación por .values (orden interno del groupby) vs out tras merges/reset_index
    size_df = grouped.size().reset_index(name="n_rows")
    out = size_df.merge(out, on=group_cols, how="left")

    # Insertar subset al inicio para que quede consistente en exports posteriores
    out.insert(0, "subset", subset_name)
    return out

# Subsets según el contrato
df_performance = df[df["is_success"] == True].copy()
df_robustez = df[df["is_valid_run"] == True].copy()

print("Subsets listos:")
print(f"- performance (is_success==True): {len(df_performance)} filas")
print(f"- robustez (is_valid_run==True): {len(df_robustez)} filas")

Subsets listos:
- performance (is_success==True): 1305 filas
- robustez (is_valid_run==True): 1500 filas


In [63]:
# 3.1 Tablas obligatorias + 3.3 Export

# Por variante (global)
stats_by_variant = pd.concat(
    [
        _agg_table(df_performance, GROUP_COLS_VARIANT, "performance"),
        _agg_table(df_robustez, GROUP_COLS_VARIANT, "robustez"),
    ],
    ignore_index=True,
    axis=0,
)

# Por tamaño (global)
stats_by_size = pd.concat(
    [
        _agg_table(df_performance, GROUP_COLS_SIZE, "performance"),
        _agg_table(df_robustez, GROUP_COLS_SIZE, "robustez"),
    ],
    ignore_index=True,
    axis=0,
)

# Por variante × tamaño
stats_by_variant_and_size = pd.concat(
    [
        _agg_table(df_performance, GROUP_COLS_VARIANT_SIZE, "performance"),
        _agg_table(df_robustez, GROUP_COLS_VARIANT_SIZE, "robustez"),
    ],
    ignore_index=True,
    axis=0,
)

# Export
OUT_VARIANT = TABLES_DIR / "stats_by_variant.csv"
OUT_SIZE = TABLES_DIR / "stats_by_size.csv"
OUT_VARIANT_SIZE = TABLES_DIR / "stats_by_variant_and_size.csv"

stats_by_variant.to_csv(OUT_VARIANT, index=False, encoding="utf-8")
stats_by_size.to_csv(OUT_SIZE, index=False, encoding="utf-8")
stats_by_variant_and_size.to_csv(OUT_VARIANT_SIZE, index=False, encoding="utf-8")

print("Export de tablas obligatorias: OK")
print(f"- {OUT_VARIANT}")
print(f"- {OUT_SIZE}")
print(f"- {OUT_VARIANT_SIZE}")

# Vista rápida
display(stats_by_variant.head(10))

Export de tablas obligatorias: OK
- D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\stats_by_variant.csv
- D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\stats_by_size.csv
- D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\stats_by_variant_and_size.csv


,subset,variant,n_rows,total_cost_mean,total_cost_median,total_cost_std,num_courses_mean,num_courses_median,num_courses_std,elapsed_time_mean,elapsed_time_median,elapsed_time_std,success_rate
0,performance,A,320,17.156250,17.0,9.164499,5.031250,5.0,2.637853,0.133481,0.000380,0.911841,1.000000
1,performance,B,320,17.156250,17.0,9.164499,5.031250,5.0,2.637853,0.130265,0.000401,0.889390,1.000000
2,performance,C,320,17.156250,17.0,9.164499,5.031250,5.0,2.637853,0.141188,0.000428,0.899543,1.000000
3,performance,D,345,26.776812,24.0,17.968789,7.576812,7.0,4.966813,12.912947,0.075614,53.553911,1.000000
4,robustez,A,375,16.386667,17.0,10.431715,4.800000,5.0,2.965939,0.113930,0.000247,0.843452,0.853333
5,robustez,B,375,16.386667,17.0,10.431715,4.800000,5.0,2.965939,0.111177,0.000251,0.822688,0.853333
6,robustez,C,375,16.386667,17.0,10.431715,4.800000,5.0,2.965939,0.120500,0.000260,0.832273,0.853333
7,robustez,D,375,24.874667,23.0,18.423357,7.050667,7.0,5.095357,11.969767,0.071098,51.489029,0.920000


In [43]:
# (opcional) stats_by_algorithm.csv

# Ojo: aquí NO estamos separando por greedy vs a_star dentro del mismo archivo de salida obligatorio.
# Esta tabla sirve para inspección rápida por algoritmo y subset.
GROUP_COLS_ALGO = ["algorithm"]
stats_by_algorithm = pd.concat(
    [
        _agg_table(df_performance, GROUP_COLS_ALGO, "performance"),
        _agg_table(df_robustez, GROUP_COLS_ALGO, "robustez"),
    ],
    ignore_index=True,
    axis=0,
)

OUT_ALGO = TABLES_DIR / "stats_by_algorithm.csv"
stats_by_algorithm.to_csv(OUT_ALGO, index=False, encoding="utf-8")

print("Export opcional: OK")
print(f"- {OUT_ALGO}")

stats_by_algorithm.sort_values(["subset", "algorithm"]).head(10)

Export opcional: OK
- D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\stats_by_algorithm.csv


,subset,n_rows,algorithm,total_cost_mean,total_cost_median,total_cost_std,num_courses_mean,num_courses_median,num_courses_std,elapsed_time_mean,elapsed_time_median,elapsed_time_std,success_rate
0,performance,560,a_star,19.194643,18.0,12.584156,5.476786,5.0,3.424028,0.249963,0.004328,1.165681,1.000000
1,performance,260,exact_small,17.807692,19.0,8.677042,5.269231,6.0,2.350679,0.014620,0.000591,0.029336,1.000000
2,performance,485,greedy,21.296907,20.0,14.700962,6.200000,6.0,4.217143,9.156217,0.000139,45.533201,1.000000
3,robustez,600,a_star,17.965000,17.0,13.009709,5.128333,5.0,3.559914,0.233421,0.003884,1.127791,0.933333
4,robustez,300,exact_small,15.533333,16.0,9.973991,4.600000,5.0,2.786734,0.012914,0.000477,0.027702,0.866667
5,robustez,600,greedy,20.540000,18.5,14.893447,5.978333,5.0,4.217920,7.457231,0.000127,41.100864,0.808333


## Figuras comparativas (exportadas)

> Normas para todas las figuras:
- Título
- Ejes con unidades
- Leyenda
- Guardado en `report/figures/` (PNG mínimo)

In [44]:
# Setup de gráficos

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

FIGURES_DIR = REPORT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Estilo consistente
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120})

def _save_fig(fig: plt.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)

print(f"Directorio de figuras: {FIGURES_DIR}")

Directorio de figuras: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\figures


In [45]:
# 4.1 Coste total por variante (boxplot) — subset is_success==True

BOXPLOT_PATH = FIGURES_DIR / "boxplot_total_cost_by_variant.png"

plot_df = df_performance.copy()
if plot_df.empty:
    raise ValueError("No hay filas en el subset performance (is_success==True). No se puede graficar el boxplot de coste.")

# Si hay varios tamaños, hacemos facetas; si no, un único axes
sizes = sorted(plot_df["instance_size"].dropna().unique().tolist())
n_sizes = max(1, len(sizes))

if n_sizes > 1:
    fig, axes = plt.subplots(1, n_sizes, figsize=(5.2 * n_sizes, 4), sharey=True)
    if n_sizes == 1:
        axes = [axes]
    for ax, size in zip(axes, sizes):
        sub = plot_df[plot_df["instance_size"] == size]
        sns.boxplot(data=sub, x="variant", y="total_cost", ax=ax)
        ax.set_title(f"Coste total por variante — tamaño: {size}")
        ax.set_xlabel("Variante")
        ax.set_ylabel("Coste total (créditos)")
    fig.suptitle("Coste total por variante (solo ejecuciones exitosas)")
else:
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.boxplot(data=plot_df, x="variant", y="total_cost", ax=ax)
    ax.set_title("Coste total por variante (solo ejecuciones exitosas)")
    ax.set_xlabel("Variante")
    ax.set_ylabel("Coste total (créditos)")

_save_fig(fig, BOXPLOT_PATH)
print(f"Figura guardada: {BOXPLOT_PATH}")

Figura guardada: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\figures\boxplot_total_cost_by_variant.png


In [64]:
# 4.2 Tasa de éxito (barplots) — tasa = promedio de is_success

SUCCESS_VARIANT_PATH = FIGURES_DIR / "success_rate_by_variant.png"
SUCCESS_VARIANT_SIZE_PATH = FIGURES_DIR / "success_rate_by_variant_and_size.png"

# Por variante
sr_variant = (
    df.groupby("variant", dropna=False)["is_success"]
    .mean()
    .reset_index()
    .rename(columns={"is_success": "success_rate"})
)
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=sr_variant, x="variant", y="success_rate", ax=ax)
ax.set_title("Tasa de éxito por variante")
ax.set_xlabel("Variante")
ax.set_ylabel("Tasa de éxito (proporción)")
ax.set_ylim(0, 1)
_save_fig(fig, SUCCESS_VARIANT_PATH)
print(f"Figura guardada: {SUCCESS_VARIANT_PATH}")

# Por variante y tamaño
sr_variant_size = (
    df.groupby(["variant", "instance_size"], dropna=False)["is_success"]
    .mean()
    .reset_index()
    .rename(columns={"is_success": "success_rate"})
)
fig, ax = plt.subplots(figsize=(7.5, 4))
sns.barplot(data=sr_variant_size, x="variant", y="success_rate", hue="instance_size", ax=ax)
ax.set_title("Tasa de éxito por variante y tamaño de instancia")
ax.set_xlabel("Variante")
ax.set_ylabel("Tasa de éxito (proporción)")
ax.set_ylim(0, 1)
ax.legend(title="Tamaño")
_save_fig(fig, SUCCESS_VARIANT_SIZE_PATH)
print(f"Figura guardada: {SUCCESS_VARIANT_SIZE_PATH}")

Figura guardada: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\figures\success_rate_by_variant.png
Figura guardada: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\figures\success_rate_by_variant_and_size.png
Figura guardada: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\figures\success_rate_by_variant_and_size.png


In [47]:
# 4.3 Tiempo de ejecución (distribución) — subset is_valid_run==True

ELAPSED_PATH = FIGURES_DIR / "elapsed_time_by_variant.png"

plot_df = df_robustez.copy()
if plot_df.empty:
    raise ValueError("No hay filas en el subset robustez (is_valid_run==True). No se puede graficar elapsed_time.")

# Heurística simple para decidir escala log (cola larga)
elapsed = plot_df["elapsed_time"].dropna()
use_log = False
if len(elapsed) >= 3:
    q50 = float(elapsed.quantile(0.50))
    q95 = float(elapsed.quantile(0.95))
    # Si el 95-percentil es muy superior a la mediana, asumimos cola larga
    if q50 > 0 and (q95 / q50) >= 20:
        use_log = True

fig, ax = plt.subplots(figsize=(6.5, 4))
sns.boxplot(data=plot_df, x="variant", y="elapsed_time", ax=ax)
ax.set_title("Tiempo de ejecución por variante (corridas válidas)")
ax.set_xlabel("Variante")
ax.set_ylabel("Tiempo (segundos)")
if use_log:
    ax.set_yscale("log")
    ax.set_ylabel("Tiempo (segundos, escala log)")

_save_fig(fig, ELAPSED_PATH)
print(f"Figura guardada: {ELAPSED_PATH}")
print(f"Escala log aplicada: {use_log}")

Figura guardada: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\figures\elapsed_time_by_variant.png
Escala log aplicada: True


In [48]:
# 4.4 Nota LLM vs coste real (scatter + correlación) — subset is_success==True AND has_llm_eval==True

LLM_SCATTER_PATH = FIGURES_DIR / "llm_nota_vs_total_cost.png"

plot_df = df_performance[df_performance["has_llm_eval"] == True].copy()
if plot_df.empty:
    raise ValueError("No hay filas con evaluación LLM dentro del subset performance (is_success==True & has_llm_eval==True).")

# Asegurar que la nota y el coste son numéricos
plot_df["llm_evaluation_nota"] = pd.to_numeric(plot_df["llm_evaluation_nota"], errors="coerce")
plot_df["total_cost"] = pd.to_numeric(plot_df["total_cost"], errors="coerce")
plot_df = plot_df.dropna(subset=["llm_evaluation_nota", "total_cost"])
if plot_df.empty:
    raise ValueError("Después de convertir a numérico, no quedaron filas válidas para llm_evaluation_nota vs total_cost.")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.scatterplot(
    data=plot_df,
    x="llm_evaluation_nota",
    y="total_cost",
    hue="instance_size",
    style="algorithm" if "algorithm" in plot_df.columns else None,
    s=90,
    ax=ax,
)
ax.set_title("Relación entre nota LLM y coste real (solo ejecuciones exitosas)")
ax.set_xlabel("Nota LLM")
ax.set_ylabel("Coste total (créditos)")
ax.legend(title="Tamaño / Algoritmo", bbox_to_anchor=(1.02, 1), loc="upper left")

# Correlaciones
x = plot_df["llm_evaluation_nota"].to_numpy()
y = plot_df["total_cost"].to_numpy()

pearson_r, pearson_p = stats.pearsonr(x, y) if len(plot_df) >= 2 else (np.nan, np.nan)
spearman_r, spearman_p = stats.spearmanr(x, y) if len(plot_df) >= 2 else (np.nan, np.nan)

# Anotar en el gráfico
txt = (
    f"Pearson r={pearson_r:.3f}, p={pearson_p:.3g}  |  "
    f"Spearman ρ={spearman_r:.3f}, p={spearman_p:.3g}  |  n={len(plot_df)}"
)
ax.text(0.02, 0.98, txt, transform=ax.transAxes, va="top", ha="left", fontsize=9)

_save_fig(fig, LLM_SCATTER_PATH)
print(f"Figura guardada: {LLM_SCATTER_PATH}")
print(txt)

Figura guardada: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\figures\llm_nota_vs_total_cost.png
Pearson r=-0.080, p=0.161  |  Spearman ρ=-0.091, p=0.109  |  n=310


## Significancia estadística (coste) con emparejamiento correcto

> Objetivo: comparar el **coste total** entre variantes en el subset **performance** (`is_success == True`).

> Reglas:
- Clave de emparejamiento: `(instance, instance_size, algorithm, seed, run)`
- Si hay pares suficientes: **Wilcoxon signed-rank** (paired)
- Si no se puede emparejar: **Mann–Whitney U** (unpaired)
- Corrección por múltiples comparaciones: **Benjamini–Hochberg (FDR)**
- Output: `report/tables/stat_tests_cost.csv`

In [49]:
# 5.1 pair_key + subset de análisis (solo performance)

PAIR_KEY_COLS = ["instance", "instance_size", "algorithm", "seed", "run"]
COST_COL = "total_cost"

test_df = df_performance.copy()
missing_pk = [c for c in PAIR_KEY_COLS if c not in test_df.columns]
if missing_pk:
    raise ValueError(f"Faltan columnas para pair_key en el subset performance: {missing_pk}")

# Asegurar que el coste sea numérico
test_df[COST_COL] = pd.to_numeric(test_df[COST_COL], errors="coerce")
test_df = test_df.dropna(subset=[COST_COL])
if test_df.empty:
    raise ValueError("El subset performance está vacío tras convertir total_cost a numérico.")

print(f"Filas para tests (performance con coste numérico): {len(test_df)}")

Filas para tests (performance con coste numérico): 1305


In [50]:
# 5.2 Comparaciones mínimas (A vs D/B/C) por instance_size + 5.3 tests + 5.4 FDR

def _benjamini_hochberg(p_values: list[float]) -> list[float]:
    """Devuelve p-values ajustados por FDR (Benjamini–Hochberg).
    Implementación simple (sin statsmodels) para mantener esto autocontenido.
    """
    m = len(p_values)
    if m == 0:
        return []
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    adj = ranked * m / (np.arange(1, m + 1))
    # Asegurar monotonicidad (desde el final)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0.0, 1.0)
    out = np.empty_like(adj)
    out[order] = adj
    return out.tolist()

def _run_comparison(df_in: pd.DataFrame, variant_a: str, variant_b: str, instance_size: str | None) -> dict:
    # Filtrar opcionalmente por tamaño
    sub = df_in if instance_size is None else df_in[df_in["instance_size"] == instance_size]
    a = sub[sub["variant"] == variant_a][PAIR_KEY_COLS + [COST_COL]].rename(columns={COST_COL: f"{COST_COL}_A"})
    b = sub[sub["variant"] == variant_b][PAIR_KEY_COLS + [COST_COL]].rename(columns={COST_COL: f"{COST_COL}_B"})

    comparison_name = f"{variant_a}_vs_{variant_b}"
    size_label = instance_size if instance_size is not None else "ALL"

    # Emparejado por intersección de claves
    merged = a.merge(b, on=PAIR_KEY_COLS, how="inner")
    n_pairs = len(merged)
    if n_pairs >= 3:
        diffs = (merged[f"{COST_COL}_B"] - merged[f"{COST_COL}_A"]).to_numpy(dtype=float)
        # Wilcoxon: requiere no todo-cero; si todo 0, el estadístico no es informativo
        if np.allclose(diffs, 0):
            return {
                "comparison": comparison_name,
                "instance_size": size_label,
                "test_name": "wilcoxon",
                "paired": True,
                "n_pairs": int(n_pairs),
                "n_a": np.nan,
                "n_b": np.nan,
                "statistic": 0.0,
                "p_value": 1.0,
                "effect": 0.0,
            }
        stat, p_val = stats.wilcoxon(diffs)
        effect = float(np.median(diffs))
        return {
            "comparison": comparison_name,
            "instance_size": size_label,
            "test_name": "wilcoxon",
            "paired": True,
            "n_pairs": int(n_pairs),
            "n_a": np.nan,
            "n_b": np.nan,
            "statistic": float(stat),
            "p_value": float(p_val),
            "effect": effect,
        }

    # Unpaired fallback: Mann–Whitney U
    vec_a = a[f"{COST_COL}_A"].to_numpy(dtype=float)
    vec_b = b[f"{COST_COL}_B"].to_numpy(dtype=float)
    n_a = int(len(vec_a))
    n_b = int(len(vec_b))
    if n_a < 2 or n_b < 2:
        # Sin datos suficientes: devolvemos fila con NaNs pero documentada
        return {
            "comparison": comparison_name,
            "instance_size": size_label,
            "test_name": "mannwhitneyu",
            "paired": False,
            "n_pairs": 0,
            "n_a": n_a,
            "n_b": n_b,
            "statistic": np.nan,
            "p_value": np.nan,
            "effect": float(np.nan),
        }

    stat, p_val = stats.mannwhitneyu(vec_a, vec_b, alternative="two-sided")
    effect = float(np.median(vec_b) - np.median(vec_a))
    return {
        "comparison": comparison_name,
        "instance_size": size_label,
        "test_name": "mannwhitneyu",
        "paired": False,
        "n_pairs": 0,
        "n_a": n_a,
        "n_b": n_b,
        "statistic": float(stat),
        "p_value": float(p_val),
        "effect": effect,
    }

# Comparaciones mínimas, por size
comparisons = [("A", "D"), ("A", "B"), ("A", "C")]
sizes = sorted(test_df["instance_size"].dropna().unique().tolist())
if not sizes:
    sizes = ["ALL"]

rows: list[dict] = []
for size in sizes:
    for va, vb in comparisons:
        rows.append(_run_comparison(test_df, va, vb, size))

stat_df = pd.DataFrame(rows)

# FDR: solo sobre pruebas con p_value numérico
p_numeric_mask = stat_df["p_value"].notna()
pvals_numeric = stat_df.loc[p_numeric_mask, "p_value"].astype(float).to_list()
pvals_fdr = _benjamini_hochberg(pvals_numeric)
stat_df["p_value_fdr"] = np.nan
stat_df.loc[p_numeric_mask, "p_value_fdr"] = pvals_fdr

# Orden legible
stat_df = stat_df[[
    "comparison",
    "instance_size",
    "test_name",
    "paired",
    "n_pairs",
    "n_a",
    "n_b",
    "statistic",
    "p_value",
    "p_value_fdr",
    "effect",
]]

display(stat_df)

,comparison,instance_size,test_name,paired,n_pairs,n_a,n_b,statistic,p_value,p_value_fdr,effect
0,A_vs_D,large,wilcoxon,True,45,NaN,NaN,0.0,4.938491e-09,1.975397e-08,21.0
1,A_vs_B,large,wilcoxon,True,45,NaN,NaN,0.0,1.000000e+00,1.000000e+00,0.0
2,A_vs_C,large,wilcoxon,True,45,NaN,NaN,0.0,1.000000e+00,1.000000e+00,0.0
3,A_vs_D,manual,wilcoxon,True,45,NaN,NaN,0.0,1.075112e-04,3.225335e-04,0.0
4,A_vs_B,manual,wilcoxon,True,45,NaN,NaN,0.0,1.000000e+00,1.000000e+00,0.0
5,A_vs_C,manual,wilcoxon,True,45,NaN,NaN,0.0,1.000000e+00,1.000000e+00,0.0
6,A_vs_D,medium,wilcoxon,True,90,NaN,NaN,0.0,4.750399e-14,2.850239e-13,7.0
7,A_vs_B,medium,wilcoxon,True,90,NaN,NaN,0.0,1.000000e+00,1.000000e+00,0.0
8,A_vs_C,medium,wilcoxon,True,90,NaN,NaN,0.0,1.000000e+00,1.000000e+00,0.0
9,A_vs_D,small,wilcoxon,True,140,NaN,NaN,307.5,2.773878e-19,3.328654e-18,4.5


In [51]:
# 5.5 Export: stat_tests_cost.csv

OUT_STAT_TESTS = TABLES_DIR / "stat_tests_cost.csv"
stat_df.to_csv(OUT_STAT_TESTS, index=False, encoding="utf-8")

print("Export de tests: OK")
print(f"- {OUT_STAT_TESTS}")
print(f"- Filas: {len(stat_df)}")

Export de tests: OK
- D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\stat_tests_cost.csv
- Filas: 12


## Análisis cualitativo usando `trajectories/*.json`

> En esta sección unimos el CSV con los JSON de `results/trajectories/` para:
- medir mejora/empeora frente a la Variante A (mismo `pair_key`)
- extraer patrones de texto sencillos (sin modelos adicionales)
- seleccionar 3–5 casos de estudio

In [52]:
# 6.1 Join CSV ↔ trajectories

from collections import Counter
import re

# Cargar trajectories
traj_rows: list[dict] = []
for path in sorted(TRAJECTORIES_DIR.glob("*.json")):
    try:
        with path.open("r", encoding="utf-8") as f:
            obj = json.load(f)
    except Exception as e:
        # En análisis preferimos fallar claro si hay archivos corruptos
        raise ValueError(f"No se pudo leer JSON: {path} ({e})") from e
    if not isinstance(obj, dict):
        raise ValueError(f"Formato inesperado en {path}: se esperaba dict, llegó {type(obj)}")
    # Guardar el path para trazabilidad
    obj = dict(obj)
    obj["trajectory_json_path"] = str(path)
    traj_rows.append(obj)

traj_df = pd.DataFrame(traj_rows)
if traj_df.empty:
    raise ValueError(f"No se cargó ningún trajectory JSON desde: {TRAJECTORIES_DIR}")

# Validar columnas clave para el join
JOIN_COLS = ["instance", "variant", "algorithm", "seed", "run"]
missing_traj_cols = [c for c in JOIN_COLS if c not in traj_df.columns]
if missing_traj_cols:
    raise ValueError(f"Trajectories sin columnas necesarias para el join: {missing_traj_cols}")

missing_csv_cols = [c for c in JOIN_COLS if c not in df.columns]
if missing_csv_cols:
    raise ValueError(f"CSV limpio sin columnas necesarias para el join: {missing_csv_cols}")

joined = df.merge(traj_df, on=JOIN_COLS, how="left", suffixes=("", "_traj"))
missing_traj = joined["trajectory_json_path"].isna().sum()
print("Join completado")
print(f"- Filas CSV: {len(df)}")
print(f"- Filas trajectories: {len(traj_df)}")
print(f"- Filas sin trajectory asociado: {missing_traj}")

# Para el cualitativo nos interesan solo corridas exitosas (para comparar costes)
joined_success = joined[joined["is_success"] == True].copy()
print(f"- Filas joined_success (is_success==True): {len(joined_success)}")

Join completado
- Filas CSV: 1500
- Filas trajectories: 1500
- Filas sin trajectory asociado: 0
- Filas joined_success (is_success==True): 1305


In [53]:
# 6.2 Mejora vs baseline (Variante A) usando el MISMO pair_key

# Construir una tabla base (Variante A) por pair_key
baseline_a = (
    joined_success[joined_success["variant"] == "A"][PAIR_KEY_COLS + ["total_cost"]]
    .rename(columns={"total_cost": "total_cost_A"})
)

# Unir baseline a todas las variantes (en success)
comp = joined_success.merge(baseline_a, on=PAIR_KEY_COLS, how="inner")
comp["delta_cost"] = pd.to_numeric(comp["total_cost"], errors="coerce") - pd.to_numeric(comp["total_cost_A"], errors="coerce")
comp = comp.dropna(subset=["delta_cost"])

# Excluir baseline en sí mismo para la clasificación improve/worsen (delta=0)
comp_non_a = comp[comp["variant"] != "A"].copy()
comp_non_a["label"] = np.where(comp_non_a["delta_cost"] < 0, "improve", np.where(comp_non_a["delta_cost"] > 0, "worsen", "tie"))

print("Comparaciones vs baseline A")
print(f"- Pares totales (incluye A): {len(comp)}")
print(f"- Pares no-A: {len(comp_non_a)}")
print(comp_non_a["label"].value_counts(dropna=False))

Comparaciones vs baseline A
- Pares totales (incluye A): 1280
- Pares no-A: 960
label
tie        700
worsen     245
improve     15
Name: count, dtype: int64


In [65]:
# 6.3 Extracción de patrones (sin modelos): tokens top por grupo

# Nota clave: el texto de LLM aparece en trajectories, pero a veces es una plantilla de error (poco informativa).
# Aquí extraemos texto desde TODOS los trajectories (joined) y generamos:
# - patrones por label (improve/worsen/tie/unlabeled)
# - un resumen de diagnósticos de texto para saber cuántas filas aportan contenido real.
from collections import Counter
import re
import json

STOPWORDS_ES = {
    "de","la","que","el","en","y","a","los","del","se","las","por","un","para","con","no","una","su","al","lo","como","más","pero","sus","le","ya","o","este","sí","porque","esta","entre","cuando","muy","sin","sobre","también","me","hasta","hay","donde","quien","desde","todo","nos","durante","todos","uno","les","ni","contra","otros","ese","eso","ante","ellos","e","esto","mí","antes","algunos","qué","unos","yo","otro","otras","otra","él","tanto","esa","estos","mucho","quienes","nada","muchos","cual","poco","ella","estar","estas","algunas","algo","nosotros","mi","mis","tú","te","ti","tu","tus",
}
STOPWORDS_EN = {
    "the","and","to","of","a","in","is","for","on","with","as","it","this","that","be","are","was","were","or","an","by","from","at","if","then","else","when","we","you","your","our","they","their","not","no",
}
STOPWORDS = STOPWORDS_ES | STOPWORDS_EN
TOKEN_RE = re.compile(r"[a-zA-ZáéíóúñÁÉÍÓÚÑ]{3,}")

MIN_TEXT_LEN = 10

def _extract_text_from_row(row: pd.Series) -> str:
    texts: list[str] = []
    # error top-level del trajectory
    if "error" in row and pd.notna(row["error"]):
        t = str(row["error"]).strip()
        if t:
            texts.append(t)
    # result.llm_evaluation.{...}
    r = row.get("result")
    if isinstance(r, dict):
        le = r.get("llm_evaluation")
        if isinstance(le, dict):
            for k in ["justification", "qualitative_comment"]:
                v = le.get(k)
                if isinstance(v, str) and v.strip():
                    texts.append(v.strip())
        # Estos campos suelen venir como dict/list, no como str. Serializar antes de usar.
        for k in ["llm_suggestion", "llm_step_log"]:
            v = r.get(k)
            if v is not None:
                if not isinstance(v, str):
                    v = json.dumps(v, ensure_ascii=False)
                if v.strip():
                    texts.append(v.strip())
    return "\n".join(texts)

def _tokenize(text: str) -> list[str]:
    toks = [t.lower() for t in TOKEN_RE.findall(text)]
    return [t for t in toks if t not in STOPWORDS]

def _text_diagnostics(df_in: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for label, sub in df_in.groupby("label", dropna=False):
        n = len(sub)
        with_text = 0
        total_len = 0
        for _, row in sub.iterrows():
            t = _extract_text_from_row(row)
            if len(t.strip()) >= MIN_TEXT_LEN:
                with_text += 1
                total_len += len(t)
        rows.append({
            "label": str(label),
            "n_rows": int(n),
            "rows_with_text": int(with_text),
            "share_with_text": float(with_text / n) if n else 0.0,
            "avg_text_len_when_present": float(total_len / with_text) if with_text else 0.0,
        })
    return pd.DataFrame(rows).sort_values(["rows_with_text", "n_rows"], ascending=False)

def _top_terms_all(df_in: pd.DataFrame, label: str, top_k: int = 30) -> pd.DataFrame:
    sub = df_in[df_in["label"] == label].copy()
    counter: Counter[str] = Counter()
    rows_with_text = 0
    for _, row in sub.iterrows():
        text = _extract_text_from_row(row)
        if len(text.strip()) >= MIN_TEXT_LEN:
            rows_with_text += 1
            counter.update(_tokenize(text))
    items = counter.most_common(top_k)
    out = pd.DataFrame({"term": [t for t,_ in items], "count": [c for _,c in items]})
    out.insert(0, "label", label)
    out.insert(1, "n_rows", len(sub))
    out.insert(2, "rows_with_text", rows_with_text)
    return out

# Base para patrones: usar joined (todas las filas)
patterns_base = joined.copy()

# Adjuntar etiqueta improve/worsen/tie cuando exista
label_map = comp_non_a[PAIR_KEY_COLS + ["variant", "label"]].drop_duplicates()
patterns_base = patterns_base.merge(label_map, on=PAIR_KEY_COLS + ["variant"], how="left")
patterns_base["label"] = patterns_base["label"].fillna("unlabeled")

diag = _text_diagnostics(patterns_base)
print("Diagnóstico de texto por label (joined):")
display(diag)

patterns_improve = _top_terms_all(patterns_base, "improve", top_k=30)
patterns_worsen = _top_terms_all(patterns_base, "worsen", top_k=30)

patterns_unlabeled = _top_terms_all(patterns_base, "unlabeled", top_k=30)
ties = _top_terms_all(patterns_base, "tie", top_k=30)

print("Patrones extraídos (top 5):")
display(patterns_improve.head(5))
display(patterns_worsen.head(5))

Diagnóstico de texto por label (joined):


,label,n_rows,rows_with_text,share_with_text,avg_text_len_when_present
1,tie,700,365,0.521429,814.334247
3,worsen,245,245,1.000000,6124.926531
2,unlabeled,540,95,0.175926,2775.863158
0,improve,15,15,1.000000,4844.666667


Patrones extraídos (top 5):


,label,n_rows,rows_with_text,term,count
0,improve,15,15,course,550
1,improve,15,15,learning,255
2,improve,15,15,python,160
3,improve,15,15,data,160
4,improve,15,15,curso,145


,label,n_rows,rows_with_text,term,count
0,worsen,245,245,course,13206
1,worsen,245,245,learning,4183
2,worsen,245,245,data,3280
3,worsen,245,245,curso,2769
4,worsen,245,245,python,2325


In [66]:
# 6.4 Export: patrones + casos de estudio (3–5)

# Exportaremos 3 archivos (como antes), pero ahora los patrones se basan en texto real extraído de trajectories.
# - llm_patterns_improve.csv: mejora vs A (si hay texto)
# - llm_patterns_worsen.csv: empeora vs A (si hay texto)
# - case_studies.csv: casos seleccionados con una columna extra 'text_excerpt' (si existe)

OUT_IMPROVE = TABLES_DIR / "llm_patterns_improve.csv"
OUT_WORSEN = TABLES_DIR / "llm_patterns_worsen.csv"
OUT_CASES = TABLES_DIR / "case_studies.csv"

# Asegurar columnas consistentes aunque no haya términos
def _ensure_patterns_schema(df_in: pd.DataFrame) -> pd.DataFrame:
    cols = ["label", "n_rows", "rows_with_text", "term", "count"]
    for c in cols:
        if c not in df_in.columns:
            df_in[c] = []
    return df_in[cols]

patterns_improve_out = _ensure_patterns_schema(patterns_improve.copy())
patterns_worsen_out = _ensure_patterns_schema(patterns_worsen.copy())

patterns_improve_out.to_csv(OUT_IMPROVE, index=False, encoding="utf-8")
patterns_worsen_out.to_csv(OUT_WORSEN, index=False, encoding="utf-8")

# Selección de casos: priorizamos improve/worsen; si faltan, completamos con ties
def _pick_cases(df_in: pd.DataFrame, label: str, n: int) -> pd.DataFrame:
    sub = df_in[df_in["label"] == label].copy()
    if sub.empty:
        return sub.head(0)
    sub["abs_delta"] = sub["delta_cost"].abs()
    sub = sub.sort_values("abs_delta", ascending=False)
    return sub.head(n)

cases = pd.concat([
    _pick_cases(comp_non_a, "improve", 2),
    _pick_cases(comp_non_a, "worsen", 2),
], ignore_index=True)

if len(cases) < 5:
    ties = comp_non_a[comp_non_a["label"] == "tie"].copy()
    if not ties.empty:
        ties["abs_delta"] = ties["delta_cost"].abs()
        ties = ties.sort_values("abs_delta", ascending=False)
        need = 5 - len(cases)
        cases = pd.concat([cases, ties.head(need)], ignore_index=True)

# Agregar excerpt de texto (si existe) desde patterns_base
text_map = patterns_base.copy()
text_map["text_excerpt"] = text_map.apply(lambda r: _extract_text_from_row(r)[:280], axis=1)
text_map = text_map[PAIR_KEY_COLS + ["variant", "text_excerpt"]]
cases = cases.merge(text_map, on=PAIR_KEY_COLS + ["variant"], how="left")

CASE_COLS = [
    "instance","instance_size","algorithm","seed","run",
    "variant","delta_cost","label",
    "total_cost","total_cost_A",
    "trajectory_json_path",
    "trajectory_path",
    "text_excerpt",
]
case_studies = cases[[c for c in CASE_COLS if c in cases.columns]].copy()
case_studies.to_csv(OUT_CASES, index=False, encoding="utf-8")

print("Exports cualitativos: OK")
print(f"- {OUT_IMPROVE} (filas: {len(patterns_improve_out)})")
print(f"- {OUT_WORSEN} (filas: {len(patterns_worsen_out)})")
print(f"- {OUT_CASES} (filas: {len(case_studies)})")

Exports cualitativos: OK
- D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\llm_patterns_improve.csv (filas: 30)
- D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\llm_patterns_worsen.csv (filas: 30)
- D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\case_studies.csv (filas: 5)


## Baseline teórico (óptimo) con `exact_small` (solo instancias small)

En este bloque calculamos el **gap al óptimo** usando como referencia el algoritmo `exact_small`, restringiendo el análisis a `instance_size == "small"`.

> Definición: para cada ejecución,
- Baseline óptimo: $\text{total\_cost\_opt}$ de la fila con `algorithm == "exact_small"` y mismas claves.
- Gap: $\text{gap} = \text{total\_cost} - \text{total\_cost\_opt}$

Se exporta:
- `report/tables/gap_to_optimal_small.csv`
- `report/figures/gap_to_optimal_by_variant_small.png`

In [67]:
# 7.1 Dominio: SOLO instance_size == "small" y baseline óptimo = algorithm == "exact_small"

GAP_OUT_PATH = TABLES_DIR / "gap_to_optimal_small.csv"
GAP_FIG_PATH = FIGURES_DIR / "gap_to_optimal_by_variant_small.png"

# Columnas mínimas para el cálculo
GAP_KEY_COLS = ["instance", "seed", "run"]
need_cols = GAP_KEY_COLS + ["variant", "algorithm", "instance_size", "total_cost"]
missing_gap_cols = [c for c in need_cols if c not in df.columns]
if missing_gap_cols:
    raise ValueError(f"CSV limpio no tiene columnas necesarias para Bloque 7: {missing_gap_cols}")

df_small = df[df["instance_size"] == "small"].copy()
if df_small.empty:
    raise ValueError("No hay filas con instance_size=='small'. No se puede calcular gap al óptimo.")

# 7.2 Baseline óptimo por ejecución: exact_small en la misma (instance, seed, run)
opt = df_small[df_small["algorithm"] == "exact_small"].copy()
if opt.empty:
    raise ValueError("No hay filas con algorithm=='exact_small' en instance_size=='small'.")

# Asegurar numericidad
df_small["total_cost"] = pd.to_numeric(df_small["total_cost"], errors="coerce")
opt["total_cost"] = pd.to_numeric(opt["total_cost"], errors="coerce")
opt = opt.dropna(subset=["total_cost"]).copy()
df_small = df_small.dropna(subset=["total_cost"]).copy()

# Si por algún motivo hay más de una fila exact_small por key, tomamos el mínimo (óptimo)
opt = (
    opt.groupby(GAP_KEY_COLS, dropna=False)["total_cost"]
    .min()
    .reset_index()
    .rename(columns={"total_cost": "total_cost_opt"})
)

gap_df = df_small.merge(opt, on=GAP_KEY_COLS, how="inner")
gap_df["gap"] = gap_df["total_cost"] - gap_df["total_cost_opt"]

# Contrato: gap por variante y ejecución (y dejamos algorithm para auditoría)
gap_out = gap_df[GAP_KEY_COLS + ["variant", "algorithm", "total_cost_opt", "total_cost", "gap"]].copy()
gap_out = gap_out.sort_values(["variant", "instance", "seed", "run", "algorithm"]).reset_index(drop=True)

gap_out.to_csv(GAP_OUT_PATH, index=False)
print("Export: gap_to_optimal_small OK")
print(f"- Archivo: {GAP_OUT_PATH}")
print(f"- Filas: {len(gap_out)}")

# 7.3 Figura: distribución del gap por variante (solo small)
fig, ax = plt.subplots(figsize=(6.5, 4))
sns.boxplot(data=gap_out, x="variant", y="gap", ax=ax)
ax.set_title("Gap al óptimo (exact_small) por variante — solo instancias small")
ax.set_xlabel("Variante")
ax.set_ylabel("Gap = total_cost - total_cost_opt")
ax.axhline(0, color="black", linewidth=1, alpha=0.8)

_save_fig(fig, GAP_FIG_PATH)
print(f"Figura guardada: {GAP_FIG_PATH}")

Export: gap_to_optimal_small OK
- Archivo: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\tables\gap_to_optimal_small.csv
- Filas: 600
Figura guardada: D:\Abraham\Escuela\3ero\2do Semestre\IA\Proyectos\Final\career-path-llm\report\figures\gap_to_optimal_by_variant_small.png
